# IND320 – Compulsory work 1: Norwegian water reservoirs

## Links
- **GitHub repository:** https://github.com/maryanahmed11011-pixel/IND320.prosjekt.maryan
- **Streamlit app:** https://FYLL-INN.streamlit.app/
- **Screencast:** FYLL INN LENKE

## AI usage
FYLL INN: Beskriv kort hvilke AI-verktøy du brukte og til hva
(f.eks. oppsett av Python-miljø og Git, feilsøking, forslag til kode for Streamlit-sidene),
og at du selv har kjørt og kontrollert koden.

## Work log (300–500 words)
FYLL INN: Skriv med egne ord om arbeidet med notebooken og Streamlit-appen:
hva du gjorde, hva som var vanskelig (f.eks. Python-miljø, Git, at dataene var i
"langt" format med flere områder), hvilke valg du tok (f.eks. å bruke totalen for Norge
og normalisere kolonnene) og hva du lærte.

## 1. Import libraries

In [ ]:
# pandas brukes til å lese og behandle tabelldata, matplotlib til plotting
import pandas as pd
import matplotlib.pyplot as plt

## 2. Read the data and rename headers to English

In [ ]:
# Les CSV-filen (ligger i data-mappen i prosjektet)
df_raw = pd.read_csv("data/reservoirs.csv")

# Oversett kolonnenavnene fra norsk til engelske, forklarende navn
rename_map = {
    "dato_Id": "date",
    "omrType": "area_type",
    "omrnr": "area_number",
    "iso_aar": "iso_year",
    "iso_uke": "iso_week",
    "fyllingsgrad": "filling_degree",
    "kapasitet_TWh": "capacity_TWh",
    "fylling_TWh": "filling_TWh",
    "neste_Publiseringsdato": "next_publication_date",
    "fyllingsgrad_forrige_uke": "filling_degree_prev_week",
    "endring_fyllingsgrad": "filling_degree_change",
}
df = df_raw.rename(columns=rename_map)

# Gjør datokolonnen om til ekte datoer, så plottene får riktig tidsakse
df["date"] = pd.to_datetime(df["date"])
df.columns.tolist()

## 3. Inspect the contents

In [ ]:
# Størrelse på datasettet (rader, kolonner) og datatyper
print("Shape:", df.shape)
df.info()

In [ ]:
# Filen inneholder flere typer områder:
#   NO   = hele Norge (area_number 0)
#   EL   = strømområdene NO1-NO5 (area_number 1-5)
#   VASS = vassdragsregioner
print(df["area_type"].value_counts())
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())

In [ ]:
# Vi bruker totalen for hele Norge, slik at vi får én ukentlig tidsserie.
# Radene i filen står ikke i datorekkefølge, så vi sorterer etter dato.
norway = df[df["area_type"] == "NO"].sort_values("date").reset_index(drop=True)

# Måleverdiene vi vil se nærmere på
measure_cols = ["filling_degree", "capacity_TWh", "filling_TWh",
                "filling_degree_prev_week", "filling_degree_change"]

# Vis de første radene og et statistisk sammendrag
display(norway.head(10))
display(norway[measure_cols].describe().round(3))

## 4. Plot each column separately

In [ ]:
# Ett delplott per måleverdi, med felles tidsakse
fig, axes = plt.subplots(len(measure_cols), 1, figsize=(12, 2.6 * len(measure_cols)), sharex=True)

for ax, col in zip(axes, measure_cols):
    ax.plot(norway["date"], norway[col], linewidth=0.8)
    ax.set_title(col)
    ax.grid(True, alpha=0.4)

axes[-1].set_xlabel("Date")
fig.suptitle("Norwegian reservoirs (national total, weekly)")
plt.tight_layout()
plt.show()

## 5. Plot all columns together
Kolonnene har ulike skalaer: kapasitet og fylling måles i TWh (titalls TWh),
mens fyllingsgrad er en andel mellom 0 og 1, og endringen er et lite tall rundt 0.
For å kunne sammenligne formen på kurvene i ett plott, min-max-normaliseres
hver kolonne til skalaen 0–1.

In [ ]:
# Min-max-normalisering: (verdi - min) / (max - min) gir 0-1 for hver kolonne
values = norway[measure_cols]
spread = (values.max() - values.min()).replace(0, 1)  # unngå deling på 0 hvis en kolonne er konstant
normalised = (values - values.min()) / spread

fig, ax = plt.subplots(figsize=(12, 5))
for col in measure_cols:
    ax.plot(norway["date"], normalised[col], linewidth=0.8, label=col)

ax.set_title("All columns together (min-max normalised)")
ax.set_xlabel("Date")
ax.set_ylabel("Normalised value (0-1)")
ax.legend(loc="upper left", fontsize=8)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Samme data for bare ett år, der sesongvariasjonen er lettere å se
one_year = norway[norway["date"].dt.year == 2023]
norm_year = (one_year[measure_cols] - values.min()) / spread

fig, ax = plt.subplots(figsize=(12, 4))
for col in measure_cols:
    ax.plot(one_year["date"], norm_year[col], marker="o", markersize=3, label=col)
ax.set_title("All columns together, 2023 (normalised)")
ax.set_xlabel("Date")
ax.set_ylabel("Normalised value (0-1)")
ax.legend(loc="upper left", fontsize=8)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()